# Eyebrow Deepfake Detection — VGG16 from Scratch + HOG + GIST + SVM

This notebook trains a **VGG16 convolutional network from random initialization** (`weights=None`), extracts its learned eyebrow features, combines them with **HOG** and **GIST-style Gabor descriptors**, and trains an **SVM** for real/fake classification.

## Drive workflow

**Input**
`AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş`

**Supporting split metadata**
`AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv`

**All experiment outputs**
`AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/<run_id>/`

## Reliability rules implemented

- Raw eyebrow images are read-only.
- Source-video IDs are recovered from the upstream selection metadata.
- Train/validation/test source-video isolation is asserted before training.
- Normalization, scalers and PCA are fitted only on the training split.
- Test data is used only after model and hyperparameter selection.
- VGG16 uses no pretrained weights; all trainable layers start randomly.
- Smoke, numerical, checkpoint and fresh-load inference tests are mandatory.
- Checkpoints and important files are written atomically.
- All figures use English text and are validated to have a minimum short edge of 600 px.
- Every run receives a unique run ID and its own output directory.

In [1]:
# Install only the libraries that may be missing in a standard Colab runtime.
# The exact environment is captured later as requirements_lock.txt.
%pip install -q scikit-image pyyaml joblib

In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import skimage
import torch
import torch.nn as nn
import yaml
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skimage.feature import hog
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import vgg16
from tqdm.auto import tqdm

warnings.filterwarnings("default")

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is designed for Google Colab because the dataset is stored in Google Drive."
    ) from exc

drive.mount("/content/drive", force_remount=False)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive


In [3]:
@dataclass
class ExperimentConfig:
    seed: int = 42
    region: str = "eyebrow"
    model_name: str = "vgg16scratch_hog_gist_svm"

    image_size: int = 128
    batch_size: int = 32
    num_workers: int = 2

    epochs: int = 30
    early_stopping_patience: int = 7
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.40
    classifier_hidden_dim: int = 256

    horizontal_flip_probability: float = 0.50
    affine_degrees: float = 5.0
    affine_translate: float = 0.03
    color_jitter_brightness: float = 0.08
    color_jitter_contrast: float = 0.08

    hog_orientations: int = 9
    hog_pixels_per_cell: Tuple[int, int] = (8, 8)
    hog_cells_per_block: Tuple[int, int] = (2, 2)

    gist_image_size: int = 128
    gist_grid_size: int = 4
    gist_orientations: int = 8
    gist_scales: Tuple[Tuple[float, float], ...] = (
        (2.0, 4.0),
        (3.0, 6.0),
        (4.0, 8.0),
        (5.0, 10.0),
    )

    pca_components: int = 256
    svm_c_values: Tuple[float, ...] = (0.1, 1.0, 10.0, 100.0)
    svm_gamma_values: Tuple[Any, ...] = ("scale", 1e-3, 1e-4)
    svm_class_weight: str = "balanced"

    checkpoint_keep_last_n_epochs: int = 3
    figure_dpi: int = 150
    minimum_figure_short_edge_px: int = 600

    # Set this only when intentionally resuming an existing run.
    resume_run_id: Optional[str] = None


CONFIG = ExperimentConfig()


def locate_drive_root() -> Path:
    candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Google Drive root could not be located after mounting.")


DRIVE_ROOT = locate_drive_root()
EXPERIMENTS_ROOT = DRIVE_ROOT / "AISC DeepFake Çalışmaları" / "Deneyler"

DATA_ROOT = EXPERIMENTS_ROOT / "Nazlıcan" / "Deney 1" / "Kaş"
ROI_METADATA_PATH = DATA_ROOT / "metadata.csv"
SELECTION_METADATA_PATH = EXPERIMENTS_ROOT / "Deney 1 Frame" / "secim_metadata.csv"
RESULTS_ROOT = EXPERIMENTS_ROOT / "Nazlıcan" / "Deney 1" / "Sonuçlar"

for required_path in (DATA_ROOT, ROI_METADATA_PATH, SELECTION_METADATA_PATH, RESULTS_ROOT):
    if not required_path.exists():
        raise FileNotFoundError(f"Required Drive path does not exist: {required_path}")

if CONFIG.resume_run_id:
    RUN_ID = CONFIG.resume_run_id
    RUN_DIR = RESULTS_ROOT / RUN_ID
    if not RUN_DIR.exists():
        raise FileNotFoundError(f"Requested resume run does not exist: {RUN_DIR}")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    RUN_ID = (
        f"{timestamp}_{CONFIG.region}_{CONFIG.model_name}_seed{CONFIG.seed}"
    )
    RUN_DIR = RESULTS_ROOT / RUN_ID
    if RUN_DIR.exists():
        raise FileExistsError(
            f"Run directory already exists: {RUN_DIR}. "
            "Wait one minute for a new run ID or set resume_run_id explicitly."
        )

OUTPUT_DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}

for path in [RUN_DIR, *OUTPUT_DIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Input data: {DATA_ROOT}")
print(f"Run ID: {RUN_ID}")
print(f"All outputs: {RUN_DIR}")

Input data: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş
Run ID: 20260806_1540_eyebrow_vgg16scratch_hog_gist_svm_seed42
All outputs: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_1540_eyebrow_vgg16scratch_hog_gist_svm_seed42


In [4]:
def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def atomic_write_text(text: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")
    temp_path.write_text(text, encoding="utf-8")
    os.replace(temp_path, target)


def atomic_write_json(payload: Dict[str, Any], target: Path) -> None:
    atomic_write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str),
        target,
    )


def atomic_write_csv(dataframe: pd.DataFrame, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")
    dataframe.to_csv(temp_path, index=False)
    os.replace(temp_path, target)


def atomic_save_checkpoint(state: Dict[str, Any], target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")
    torch.save(state, temp_path)

    loaded = torch.load(temp_path, map_location="cpu", weights_only=False)
    required_keys = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "best_metric_score",
        "config",
        "torch_rng_state",
        "numpy_rng_state",
        "python_rng_state",
    }
    missing_keys = required_keys.difference(loaded)
    if missing_keys:
        raise RuntimeError(
            f"Checkpoint integrity verification failed. Missing keys: {sorted(missing_keys)}"
        )

    os.replace(temp_path, target)


def atomic_joblib_dump(payload: Any, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")
    joblib.dump(payload, temp_path)
    _ = joblib.load(temp_path)
    os.replace(temp_path, target)


def atomic_save_npz(target: Path, **arrays: Any) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")
    with temp_path.open("wb") as file_handle:
        np.savez_compressed(file_handle, **arrays)

    with np.load(temp_path, allow_pickle=False) as loaded:
        if not loaded.files:
            raise RuntimeError(f"NPZ integrity verification failed: {temp_path}")

    os.replace(temp_path, target)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def save_figure(fig: plt.Figure, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(target, dpi=CONFIG.figure_dpi, bbox_inches="tight")
    plt.close(fig)

    with Image.open(target) as image:
        if min(image.size) < CONFIG.minimum_figure_short_edge_px:
            raise AssertionError(
                f"Figure resolution is below the minimum: {target} -> {image.size}"
            )


def clean_for_yaml(value: Any) -> Any:
    if isinstance(value, tuple):
        return [clean_for_yaml(item) for item in value]
    if isinstance(value, dict):
        return {key: clean_for_yaml(item) for key, item in value.items()}
    return value


set_global_seed(CONFIG.seed)

resolved_config = {
    **clean_for_yaml(asdict(CONFIG)),
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "roi_metadata_path": str(ROI_METADATA_PATH),
    "selection_metadata_path": str(SELECTION_METADATA_PATH),
    "results_root": str(RESULTS_ROOT),
    "run_dir": str(RUN_DIR),
    "created_at": datetime.now().isoformat(),
}

atomic_write_text(
    yaml.safe_dump(resolved_config, allow_unicode=True, sort_keys=False),
    RUN_DIR / "config_resolved.yaml",
)

## 1. Metadata resolution and quality gates

The eyebrow ROI metadata does not itself contain usable source-video IDs.  
The notebook joins it with `Deney 1 Frame/secim_metadata.csv` using the renamed frame filename, then derives the source video from the original frame path.

Training is blocked if:

- mandatory metadata columns are missing,
- a successful image is absent,
- sample IDs are duplicated,
- file hashes are duplicated across splits,
- source-video IDs are missing,
- or a source video appears in more than one split.

In [5]:
ROI_REQUIRED_COLUMNS = {
    "sample_id",
    "frame_index",
    "face_index",
    "label",
    "split",
    "status",
    "skip_reason",
    "output_path",
    "output_relative_path",
    "output_sha256",
    "run_id",
}

SELECTION_REQUIRED_COLUMNS = {
    "sinif",
    "split",
    "orijinal_yol",
    "yeni_yol",
    "dosya_adi",
}

roi_metadata = pd.read_csv(ROI_METADATA_PATH)
selection_metadata = pd.read_csv(SELECTION_METADATA_PATH)

missing_roi_columns = ROI_REQUIRED_COLUMNS.difference(roi_metadata.columns)
missing_selection_columns = SELECTION_REQUIRED_COLUMNS.difference(
    selection_metadata.columns
)

if missing_roi_columns:
    raise ValueError(
        f"ROI metadata is missing required columns: {sorted(missing_roi_columns)}"
    )
if missing_selection_columns:
    raise ValueError(
        "Selection metadata is missing required columns: "
        f"{sorted(missing_selection_columns)}"
    )

total_inputs = len(roi_metadata)
success_count = int((roi_metadata["status"] == "SUCCESS").sum())
skipped_count = int((roi_metadata["status"] == "SKIPPED").sum())
error_count = int((roi_metadata["status"] == "ERROR").sum())

assert total_inputs == success_count + skipped_count + error_count, (
    "Input accounting mismatch: total != success + skipped + error"
)
assert roi_metadata["sample_id"].is_unique, "Duplicate sample_id detected."
assert roi_metadata["output_path"].notna().all(), "Missing output_path detected."

selection_resolved = selection_metadata.copy()
selection_resolved["file_name"] = selection_resolved["dosya_adi"].astype(str)
selection_resolved["label_from_selection"] = (
    selection_resolved["sinif"].astype(str).str.lower()
)
selection_resolved["split_from_selection"] = (
    selection_resolved["split"].astype(str).str.lower()
)
selection_resolved["source_video_raw"] = selection_resolved["orijinal_yol"].map(
    lambda value: Path(str(value)).parent.name
)
selection_resolved["source_video"] = (
    selection_resolved["label_from_selection"]
    + "::"
    + selection_resolved["source_video_raw"].astype(str)
)
selection_resolved["source_frame_index"] = selection_resolved[
    "orijinal_yol"
].map(
    lambda value: int(re.search(r"(\d+)$", Path(str(value)).stem).group(1))
    if re.search(r"(\d+)$", Path(str(value)).stem)
    else -1
)

if selection_resolved["file_name"].duplicated().any():
    duplicates = selection_resolved.loc[
        selection_resolved["file_name"].duplicated(keep=False), "file_name"
    ].head(10)
    raise ValueError(
        "Selection metadata contains duplicate renamed filenames: "
        f"{duplicates.tolist()}"
    )

metadata = roi_metadata.copy()
metadata["file_name"] = metadata["output_relative_path"].map(
    lambda value: Path(str(value)).name
)

# ROI metadata already contains a source_video column, but in the current
# eyebrow metadata that column is empty. Use an explicit resolved column name
# during the merge to prevent pandas from creating source_video_x/source_video_y.
selection_for_merge = selection_resolved[
    [
        "file_name",
        "label_from_selection",
        "split_from_selection",
        "source_video",
        "source_video_raw",
        "source_frame_index",
        "orijinal_yol",
    ]
].rename(columns={"source_video": "resolved_source_video"})

metadata = metadata.merge(
    selection_for_merge,
    on="file_name",
    how="left",
    validate="one_to_one",
)

if "source_video" not in metadata.columns:
    metadata["source_video"] = np.nan

metadata["source_video"] = metadata["resolved_source_video"].combine_first(
    metadata["source_video"]
)
metadata = metadata.drop(columns=["resolved_source_video"])

if metadata["source_video"].isna().any():
    unresolved = metadata.loc[
        metadata["source_video"].isna(), "file_name"
    ].head(20)
    raise ValueError(
        "Source-video recovery failed for these files: "
        f"{unresolved.tolist()}"
    )

label_mismatch = metadata[
    metadata["label"].str.lower() != metadata["label_from_selection"]
]
split_mismatch = metadata[
    metadata["split"].str.lower() != metadata["split_from_selection"]
]

if not label_mismatch.empty:
    raise AssertionError(
        f"Label mismatch detected in {len(label_mismatch)} metadata rows."
    )
if not split_mismatch.empty:
    raise AssertionError(
        f"Split mismatch detected in {len(split_mismatch)} metadata rows."
    )

# Eyebrow ROI does not have an open/closed/talking state.
metadata["roi_state"] = "not_applicable"
metadata["resolved_output_path"] = metadata["output_relative_path"].map(
    lambda value: str(DATA_ROOT / Path(str(value)))
)

successful_metadata = metadata[metadata["status"] == "SUCCESS"].copy()
successful_metadata["resolved_output_path"] = successful_metadata[
    "resolved_output_path"
].map(str)

missing_success_files = [
    path
    for path in successful_metadata["resolved_output_path"]
    if not Path(path).exists()
]
if missing_success_files:
    raise FileNotFoundError(
        "SUCCESS rows point to missing files. Examples: "
        f"{missing_success_files[:10]}"
    )

if successful_metadata["output_sha256"].isna().any():
    raise AssertionError("A SUCCESS row has a missing output_sha256.")

if successful_metadata["output_sha256"].duplicated().any():
    duplicate_hashes = successful_metadata.loc[
        successful_metadata["output_sha256"].duplicated(keep=False),
        ["file_name", "output_sha256", "split"],
    ].head(20)
    raise AssertionError(
        "Duplicate successful image hashes detected:\n"
        f"{duplicate_hashes.to_string(index=False)}"
    )

allowed_labels = {"real", "fake"}
allowed_splits = {"train", "val", "test"}

if set(successful_metadata["label"].str.lower()) != allowed_labels:
    raise AssertionError(
        f"Expected labels {allowed_labels}, found "
        f"{set(successful_metadata['label'].str.lower())}"
    )

if set(successful_metadata["split"].str.lower()) != allowed_splits:
    raise AssertionError(
        f"Expected splits {allowed_splits}, found "
        f"{set(successful_metadata['split'].str.lower())}"
    )

successful_metadata["label"] = successful_metadata["label"].str.lower()
successful_metadata["split"] = successful_metadata["split"].str.lower()
successful_metadata["target"] = successful_metadata["label"].map(
    {"real": 0, "fake": 1}
)

split_video_ids = {
    split_name: set(
        successful_metadata.loc[
            successful_metadata["split"] == split_name, "source_video"
        ]
    )
    for split_name in ("train", "val", "test")
}

assert split_video_ids["train"].isdisjoint(split_video_ids["val"]), (
    "Source-video leakage exists between train and validation."
)
assert split_video_ids["train"].isdisjoint(split_video_ids["test"]), (
    "Source-video leakage exists between train and test."
)
assert split_video_ids["val"].isdisjoint(split_video_ids["test"]), (
    "Source-video leakage exists between validation and test."
)

hashes_by_split = {
    split_name: set(
        successful_metadata.loc[
            successful_metadata["split"] == split_name, "output_sha256"
        ]
    )
    for split_name in ("train", "val", "test")
}

assert hashes_by_split["train"].isdisjoint(hashes_by_split["val"]), (
    "Identical image content exists in train and validation."
)
assert hashes_by_split["train"].isdisjoint(hashes_by_split["test"]), (
    "Identical image content exists in train and test."
)
assert hashes_by_split["val"].isdisjoint(hashes_by_split["test"]), (
    "Identical image content exists in validation and test."
)

metadata_columns = [
    "sample_id",
    "source_video",
    "source_video_raw",
    "source_frame_index",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "target",
    "split",
    "status",
    "skip_reason",
    "output_sha256",
    "resolved_output_path",
    "run_id",
]
metadata_used = successful_metadata[metadata_columns].copy()
atomic_write_csv(metadata_used, OUTPUT_DIRS["artifacts"] / "metadata_used.csv")

accounting_payload = {
    "run_id": RUN_ID,
    "total_inputs": total_inputs,
    "success_count": success_count,
    "skipped_count": skipped_count,
    "error_count": error_count,
    "training_eligible_success_count": len(metadata_used),
    "unique_source_videos": int(metadata_used["source_video"].nunique()),
    "split_class_counts": (
        metadata_used.groupby(["split", "label"]).size().unstack(fill_value=0)
    ).to_dict(orient="index"),
    "split_source_video_counts": (
        metadata_used.groupby("split")["source_video"].nunique().to_dict()
    ),
    "source_video_intersections": {
        "train_val": len(split_video_ids["train"] & split_video_ids["val"]),
        "train_test": len(split_video_ids["train"] & split_video_ids["test"]),
        "val_test": len(split_video_ids["val"] & split_video_ids["test"]),
    },
}

atomic_write_json(
    accounting_payload,
    OUTPUT_DIRS["metrics"] / "data_accounting.json",
)

print(json.dumps(accounting_payload, ensure_ascii=False, indent=2))

{
  "run_id": "20260806_1540_eyebrow_vgg16scratch_hog_gist_svm_seed42",
  "total_inputs": 3000,
  "success_count": 1962,
  "skipped_count": 1038,
  "error_count": 0,
  "training_eligible_success_count": 1962,
  "unique_source_videos": 1429,
  "split_class_counts": {
    "test": {
      "fake": 95,
      "real": 101
    },
    "train": {
      "fake": 776,
      "real": 786
    },
    "val": {
      "fake": 98,
      "real": 106
    }
  },
  "split_source_video_counts": {
    "test": 138,
    "train": 1145,
    "val": 146
  },
  "source_video_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  }
}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# Save an English, publication-ready dataset distribution figure.
distribution = (
    metadata_used.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
distribution.plot(kind="bar", ax=ax)
ax.set_title("Successful Eyebrow ROI Samples by Split and Class", fontsize=14, fontweight="bold")
ax.set_xlabel("Dataset Split", fontsize=11)
ax.set_ylabel("Number of Samples", fontsize=11)
ax.legend(title="Class")
ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, OUTPUT_DIRS["figures"] / "dataset_distribution.png")

display(distribution)

label,fake,real
split,,
train,776,786
val,98,106
test,95,101


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Train-only image statistics and data loaders

Images are letterboxed instead of stretched.  
RGB mean and standard deviation are computed **only from the training split** and are then reused for validation and test.

In [7]:
class Letterbox:
    def __init__(self, size: int, fill: Tuple[int, int, int] = (0, 0, 0)):
        self.size = int(size)
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        return ImageOps.pad(
            image.convert("RGB"),
            (self.size, self.size),
            method=Image.Resampling.BILINEAR,
            color=self.fill,
            centering=(0.5, 0.5),
        )


class EyebrowDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform: transforms.Compose,
    ) -> None:
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        row = self.dataframe.iloc[index]
        image_path = Path(row["resolved_output_path"])

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                tensor = self.transform(image)
        except Exception as exc:
            raise RuntimeError(f"Failed to read image: {image_path}") from exc

        return {
            "image": tensor,
            "target": torch.tensor(int(row["target"]), dtype=torch.long),
            "sample_id": str(row["sample_id"]),
            "source_video": str(row["source_video"]),
            "path": str(image_path),
        }


base_stat_transform = transforms.Compose(
    [
        Letterbox(CONFIG.image_size),
        transforms.ToTensor(),
    ]
)

train_frame = metadata_used[metadata_used["split"] == "train"].copy()
val_frame = metadata_used[metadata_used["split"] == "val"].copy()
test_frame = metadata_used[metadata_used["split"] == "test"].copy()

stat_dataset = EyebrowDataset(train_frame, base_stat_transform)
stat_loader = DataLoader(
    stat_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
)

channel_sum = torch.zeros(3, dtype=torch.float64)
channel_squared_sum = torch.zeros(3, dtype=torch.float64)
pixel_count = 0

for batch in tqdm(stat_loader, desc="Computing train-only RGB statistics"):
    images = batch["image"].to(dtype=torch.float64)
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_squared_sum += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.shape[0] * images.shape[2] * images.shape[3]

train_mean = channel_sum / pixel_count
train_variance = channel_squared_sum / pixel_count - train_mean ** 2
train_std = torch.sqrt(torch.clamp(train_variance, min=1e-12))

TRAIN_MEAN = tuple(float(value) for value in train_mean)
TRAIN_STD = tuple(float(value) for value in train_std)

atomic_write_json(
    {
        "mean": TRAIN_MEAN,
        "std": TRAIN_STD,
        "computed_from_split": "train",
        "sample_count": len(train_frame),
    },
    OUTPUT_DIRS["artifacts"] / "train_normalization.json",
)

print("Train-only mean:", TRAIN_MEAN)
print("Train-only std:", TRAIN_STD)

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=1469) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Computing train-only RGB statistics:   0%|          | 0/49 [00:00<?, ?it/s]

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7c7371f2ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7c720c65c520>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7c7371f2ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7c720c65c520>
  self.pid = os.fork()


Train-only mean: (0.15145024263925405, 0.11068506679243253, 0.09534352670458353)
Train-only std: (0.2856628378428359, 0.21304641710489225, 0.18695454305386672)


In [8]:
train_transform = transforms.Compose(
    [
        Letterbox(CONFIG.image_size),
        transforms.RandomHorizontalFlip(
            p=CONFIG.horizontal_flip_probability
        ),
        transforms.RandomAffine(
            degrees=CONFIG.affine_degrees,
            translate=(
                CONFIG.affine_translate,
                CONFIG.affine_translate,
            ),
            fill=0,
        ),
        transforms.ColorJitter(
            brightness=CONFIG.color_jitter_brightness,
            contrast=CONFIG.color_jitter_contrast,
        ),
        transforms.ToTensor(),
        transforms.Normalize(TRAIN_MEAN, TRAIN_STD),
    ]
)

evaluation_transform = transforms.Compose(
    [
        Letterbox(CONFIG.image_size),
        transforms.ToTensor(),
        transforms.Normalize(TRAIN_MEAN, TRAIN_STD),
    ]
)

train_dataset = EyebrowDataset(train_frame, train_transform)
val_dataset = EyebrowDataset(val_frame, evaluation_transform)
test_dataset = EyebrowDataset(test_frame, evaluation_transform)

loader_generator = torch.Generator()
loader_generator.manual_seed(CONFIG.seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=True,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator,
    persistent_workers=CONFIG.num_workers > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CONFIG.num_workers > 0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CONFIG.num_workers > 0,
)

print(
    {
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "test_samples": len(test_dataset),
    }
)

{'train_samples': 1562, 'val_samples': 204, 'test_samples': 196}


## 3. VGG16 trained from scratch

`torchvision.models.vgg16(weights=None)` constructs the architecture without downloading or loading pretrained parameters.

The convolutional backbone is VGG16. Its original large ImageNet classifier is replaced with a smaller classifier suitable for this eyebrow dataset. All parameters remain trainable.

In [9]:
class VGG16ScratchClassifier(nn.Module):
    def __init__(
        self,
        hidden_dim: int,
        dropout: float,
        number_of_classes: int = 2,
    ) -> None:
        super().__init__()

        network = vgg16(weights=None)
        self.features = network.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, number_of_classes),
        )

    def extract_features(self, images: torch.Tensor) -> torch.Tensor:
        feature_maps = self.features(images)
        pooled = self.avgpool(feature_maps)
        return torch.flatten(pooled, 1)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.extract_features(images)
        return self.classifier(features)


def build_model() -> VGG16ScratchClassifier:
    model = VGG16ScratchClassifier(
        hidden_dim=CONFIG.classifier_hidden_dim,
        dropout=CONFIG.dropout,
        number_of_classes=2,
    )

    if not all(parameter.requires_grad for parameter in model.parameters()):
        raise AssertionError("All VGG16 parameters must be trainable.")

    return model


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

model = build_model().to(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

model_payload = {
    "architecture": "VGG16 convolutional backbone with adaptive global pooling",
    "pretrained_weights": False,
    "torchvision_weights_argument": None,
    "all_parameters_trainable": True,
    "parameter_count": parameter_count,
    "trainable_parameter_count": trainable_parameter_count,
    "device": str(DEVICE),
}

atomic_write_json(
    model_payload,
    OUTPUT_DIRS["artifacts"] / "model_architecture.json",
)

print(json.dumps(model_payload, indent=2))

{
  "architecture": "VGG16 convolutional backbone with adaptive global pooling",
  "pretrained_weights": false,
  "torchvision_weights_argument": null,
  "all_parameters_trainable": true,
  "parameter_count": 14846530,
  "trainable_parameter_count": 14846530,
  "device": "cuda"
}


## 4. Mandatory quality gates before full training

The following cell performs:

1. two-batch forward/backward smoke testing,
2. NaN/Inf checks for loss and gradients,
3. atomic checkpoint save/load testing,
4. output equivalence after loading the checkpoint into a fresh model.

In [10]:
def compute_class_weights(frame: pd.DataFrame) -> torch.Tensor:
    counts = (
        frame["target"]
        .value_counts()
        .reindex([0, 1])
        .astype(float)
        .to_numpy()
    )
    if np.any(counts <= 0):
        raise ValueError(f"Invalid class counts: {counts.tolist()}")

    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


CLASS_WEIGHTS = compute_class_weights(train_frame)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)


def assert_finite_gradients(candidate_model: nn.Module) -> None:
    for name, parameter in candidate_model.named_parameters():
        if parameter.grad is None:
            continue
        if not torch.isfinite(parameter.grad).all():
            raise FloatingPointError(
                f"NaN or Inf gradient detected in parameter: {name}"
            )


def run_quality_gates() -> None:
    smoke_model = build_model().to(DEVICE)
    smoke_optimizer = torch.optim.AdamW(
        smoke_model.parameters(),
        lr=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
    )

    smoke_model.train()
    inspected_batches = []

    for batch_index, batch in enumerate(train_loader):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        smoke_optimizer.zero_grad(set_to_none=True)
        logits = smoke_model(images)
        loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Smoke test produced a non-finite loss at batch {batch_index}."
            )

        loss.backward()
        assert_finite_gradients(smoke_model)
        smoke_optimizer.step()

        inspected_batches.append(
            {
                "batch_index": batch_index,
                "loss": float(loss.detach().cpu()),
                "batch_size": int(images.shape[0]),
            }
        )

        if batch_index == 1:
            break

    if len(inspected_batches) != 2:
        raise RuntimeError("Smoke test could not inspect two batches.")

    checkpoint_test_path = (
        OUTPUT_DIRS["checkpoints"] / "checkpoint_quality_gate.ckpt"
    )

    checkpoint_state = {
        "epoch": 0,
        "model_state_dict": smoke_model.state_dict(),
        "optimizer_state_dict": smoke_optimizer.state_dict(),
        "scheduler_state_dict": {},
        "scaler_state_dict": {},
        "best_metric_score": 0.0,
        "config": resolved_config,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }
    atomic_save_checkpoint(checkpoint_state, checkpoint_test_path)

    verification_batch = next(iter(val_loader))
    verification_images = verification_batch["image"].to(
        DEVICE, non_blocking=True
    )
    verification_targets = verification_batch["target"].to(
        DEVICE, non_blocking=True
    )

    smoke_model.eval()
    with torch.no_grad():
        original_logits = smoke_model(verification_images)
        original_loss = criterion(
            original_logits, verification_targets
        ).item()

    fresh_model = build_model().to(DEVICE)
    loaded_checkpoint = torch.load(
        checkpoint_test_path,
        map_location=DEVICE,
        weights_only=False,
    )
    fresh_model.load_state_dict(loaded_checkpoint["model_state_dict"])
    fresh_model.eval()

    with torch.no_grad():
        loaded_logits = fresh_model(verification_images)
        loaded_loss = criterion(
            loaded_logits, verification_targets
        ).item()

    if not torch.allclose(
        original_logits,
        loaded_logits,
        rtol=1e-6,
        atol=1e-7,
    ):
        raise AssertionError(
            "Freshly loaded checkpoint does not reproduce model outputs."
        )

    if not math.isclose(
        original_loss,
        loaded_loss,
        rel_tol=1e-6,
        abs_tol=1e-7,
    ):
        raise AssertionError(
            "Freshly loaded checkpoint does not reproduce the same loss."
        )

    quality_gate_payload = {
        "status": "PASSED",
        "smoke_batches": inspected_batches,
        "checkpoint_path": str(checkpoint_test_path),
        "original_loss": original_loss,
        "loaded_loss": loaded_loss,
        "output_equivalence": True,
        "tested_at": datetime.now().isoformat(),
    }

    atomic_write_json(
        quality_gate_payload,
        OUTPUT_DIRS["metrics"] / "quality_gates.json",
    )

    del smoke_model, fresh_model, smoke_optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


run_quality_gates()
print("All pre-training quality gates passed.")

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=1469) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


All pre-training quality gates passed.


## 5. Train the scratch VGG16 classifier

- Selection metric: validation F1.
- `best.ckpt`: best validation F1.
- `last.ckpt`: latest completed epoch.
- Recent epoch checkpoints are rotated.
- Early stopping is based only on validation data.

In [11]:
def binary_metrics(
    targets: np.ndarray,
    predictions: np.ndarray,
    scores: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    metrics = {
        "accuracy": float(accuracy_score(targets, predictions)),
        "precision": float(
            precision_score(
                targets,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                targets,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                targets,
                predictions,
                zero_division=0,
            )
        ),
    }

    matrix = confusion_matrix(targets, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = matrix.ravel()
    specificity_denominator = true_negative + false_positive
    metrics["specificity"] = float(
        true_negative / specificity_denominator
        if specificity_denominator
        else 0.0
    )

    if scores is not None and len(np.unique(targets)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(targets, scores))
        metrics["average_precision"] = float(
            average_precision_score(targets, scores)
        )

    return metrics


def run_cnn_epoch(
    candidate_model: nn.Module,
    loader: DataLoader,
    optimizer: Optional[torch.optim.Optimizer],
    use_amp: bool,
    scaler: Optional[torch.amp.GradScaler],
) -> Tuple[float, Dict[str, float]]:
    training = optimizer is not None
    candidate_model.train(training)

    losses: List[float] = []
    all_targets: List[int] = []
    all_predictions: List[int] = []
    all_scores: List[float] = []

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=use_amp,
            ):
                logits = candidate_model(images)
                loss = criterion(logits, targets)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "NaN or Inf loss detected during training/evaluation."
                )

            if training:
                if scaler is None:
                    loss.backward()
                    assert_finite_gradients(candidate_model)
                    optimizer.step()
                else:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    assert_finite_gradients(candidate_model)
                    scaler.step(optimizer)
                    scaler.update()

        probabilities = torch.softmax(logits.detach(), dim=1)[:, 1]
        predictions = torch.argmax(logits.detach(), dim=1)

        losses.append(float(loss.detach().cpu()))
        all_targets.extend(targets.detach().cpu().numpy().astype(int).tolist())
        all_predictions.extend(
            predictions.detach().cpu().numpy().astype(int).tolist()
        )
        all_scores.extend(
            probabilities.detach().cpu().numpy().astype(float).tolist()
        )

    epoch_loss = float(np.mean(losses))
    epoch_metrics = binary_metrics(
        np.asarray(all_targets),
        np.asarray(all_predictions),
        np.asarray(all_scores),
    )
    return epoch_loss, epoch_metrics


def make_checkpoint_state(
    epoch: int,
    candidate_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    grad_scaler: Optional[torch.amp.GradScaler],
    best_metric_score: float,
    history: List[Dict[str, Any]],
) -> Dict[str, Any]:
    return {
        "epoch": epoch,
        "model_state_dict": candidate_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": (
            grad_scaler.state_dict() if grad_scaler is not None else {}
        ),
        "best_metric_score": best_metric_score,
        "history": history,
        "config": resolved_config,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }


def rotate_epoch_checkpoints(checkpoint_directory: Path) -> None:
    epoch_paths = sorted(
        checkpoint_directory.glob("epoch_*.ckpt"),
        key=lambda path: int(
            re.search(r"epoch_(\d+)", path.stem).group(1)
        ),
    )

    excess_count = (
        len(epoch_paths) - CONFIG.checkpoint_keep_last_n_epochs
    )
    if excess_count <= 0:
        return

    for path in epoch_paths[:excess_count]:
        path.unlink()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

USE_AMP = DEVICE.type == "cuda"
grad_scaler = (
    torch.amp.GradScaler("cuda", enabled=True)
    if USE_AMP
    else None
)

best_metric_score = -np.inf
start_epoch = 1
history: List[Dict[str, Any]] = []
epochs_without_improvement = 0

last_checkpoint_path = OUTPUT_DIRS["checkpoints"] / "last.ckpt"
best_checkpoint_path = OUTPUT_DIRS["checkpoints"] / "best.ckpt"

if CONFIG.resume_run_id and last_checkpoint_path.exists():
    checkpoint = torch.load(
        last_checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if grad_scaler is not None and checkpoint.get("scaler_state_dict"):
        grad_scaler.load_state_dict(checkpoint["scaler_state_dict"])

    best_metric_score = float(checkpoint["best_metric_score"])
    history = list(checkpoint.get("history", []))
    start_epoch = int(checkpoint["epoch"]) + 1

    torch.set_rng_state(checkpoint["torch_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    random.setstate(checkpoint["python_rng_state"])

    if torch.cuda.is_available() and checkpoint.get("cuda_rng_state"):
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(f"Resuming from epoch {start_epoch}.")

training_started_at = time.time()

for epoch in range(start_epoch, CONFIG.epochs + 1):
    train_loss, train_metrics = run_cnn_epoch(
        model,
        train_loader,
        optimizer,
        USE_AMP,
        grad_scaler,
    )
    val_loss, val_metrics = run_cnn_epoch(
        model,
        val_loader,
        optimizer=None,
        use_amp=False,
        scaler=None,
    )

    validation_f1 = val_metrics["f1"]
    scheduler.step(validation_f1)

    epoch_record = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "train_loss": train_loss,
        "val_loss": val_loss,
        **{
            f"train_{key}": value
            for key, value in train_metrics.items()
        },
        **{
            f"val_{key}": value
            for key, value in val_metrics.items()
        },
    }
    history.append(epoch_record)

    improved = validation_f1 > best_metric_score + 1e-12
    if improved:
        best_metric_score = validation_f1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    checkpoint_state = make_checkpoint_state(
        epoch,
        model,
        optimizer,
        scheduler,
        grad_scaler,
        best_metric_score,
        history,
    )

    atomic_save_checkpoint(checkpoint_state, last_checkpoint_path)
    atomic_save_checkpoint(
        checkpoint_state,
        OUTPUT_DIRS["checkpoints"] / f"epoch_{epoch}.ckpt",
    )
    rotate_epoch_checkpoints(OUTPUT_DIRS["checkpoints"])

    if improved:
        atomic_save_checkpoint(checkpoint_state, best_checkpoint_path)

    atomic_write_csv(
        pd.DataFrame(history),
        OUTPUT_DIRS["logs"] / "training_history.csv",
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1={validation_f1:.4f} | "
        f"best_val_f1={best_metric_score:.4f}"
    )

    if epochs_without_improvement >= CONFIG.early_stopping_patience:
        print(
            "Early stopping activated after "
            f"{epochs_without_improvement} epochs without improvement."
        )
        break

training_duration_seconds = time.time() - training_started_at

if not best_checkpoint_path.exists():
    raise FileNotFoundError("best.ckpt was not created.")

atomic_write_json(
    {
        "training_duration_seconds": training_duration_seconds,
        "epochs_completed": len(history),
        "best_validation_f1": best_metric_score,
        "best_checkpoint": str(best_checkpoint_path),
        "last_checkpoint": str(last_checkpoint_path),
    },
    OUTPUT_DIRS["metrics"] / "cnn_training_summary.json",
)

Epoch 01 | train_loss=0.6937 | val_loss=0.6931 | val_f1=0.5345 | best_val_f1=0.5345


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch 02 | train_loss=0.6929 | val_loss=0.6978 | val_f1=0.6490 | best_val_f1=0.6490


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch 03 | train_loss=0.6934 | val_loss=0.6937 | val_f1=0.6490 | best_val_f1=0.6490


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch 08 | train_loss=0.6923 | val_loss=0.6876 | val_f1=0.0000 | best_val_f1=0.6490


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch 09 | train_loss=0.6921 | val_loss=0.6919 | val_f1=0.4918 | best_val_f1=0.6490
Early stopping activated after 7 epochs without improvement.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
history_frame = pd.DataFrame(history)
if history_frame.empty:
    raise RuntimeError("Training history is empty.")

fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
ax.plot(
    history_frame["epoch"],
    history_frame["train_loss"],
    label="Training Loss",
    linewidth=2,
)
ax.plot(
    history_frame["epoch"],
    history_frame["val_loss"],
    label="Validation Loss",
    linewidth=2,
    linestyle="--",
)
ax.set_title("VGG16 Training and Validation Loss", fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Loss", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(fig, OUTPUT_DIRS["figures"] / "cnn_loss_curve.png")

fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
ax.plot(
    history_frame["epoch"],
    history_frame["train_accuracy"],
    label="Training Accuracy",
    linewidth=2,
)
ax.plot(
    history_frame["epoch"],
    history_frame["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2,
    linestyle="--",
)
ax.set_title("VGG16 Training and Validation Accuracy", fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(fig, OUTPUT_DIRS["figures"] / "cnn_accuracy_curve.png")

fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
ax.plot(
    history_frame["epoch"],
    history_frame["train_f1"],
    label="Training F1",
    linewidth=2,
)
ax.plot(
    history_frame["epoch"],
    history_frame["val_f1"],
    label="Validation F1",
    linewidth=2,
    linestyle="--",
)
ax.set_title("VGG16 Training and Validation F1 Score", fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(fig, OUTPUT_DIRS["figures"] / "cnn_f1_curve.png")

## 6. Fresh-load inference test and learned VGG16 feature extractor

The best scratch-trained checkpoint is loaded into a newly constructed model.  
That model must successfully produce predictions before feature extraction is allowed.

In [13]:
best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

feature_model = build_model().to(DEVICE)
feature_model.load_state_dict(best_checkpoint["model_state_dict"])
feature_model.eval()

inference_batch = next(iter(val_loader))
with torch.no_grad():
    inference_logits = feature_model(
        inference_batch["image"].to(DEVICE)
    )

if inference_logits.shape[1] != 2:
    raise AssertionError(
        f"Unexpected inference output shape: {tuple(inference_logits.shape)}"
    )
if not torch.isfinite(inference_logits).all():
    raise FloatingPointError(
        "Freshly loaded best checkpoint produced NaN or Inf outputs."
    )

atomic_write_json(
    {
        "status": "PASSED",
        "checkpoint": str(best_checkpoint_path),
        "batch_size": int(inference_logits.shape[0]),
        "output_shape": list(inference_logits.shape),
        "tested_at": datetime.now().isoformat(),
    },
    OUTPUT_DIRS["metrics"] / "fresh_load_inference_test.json",
)

print("Fresh-load inference test passed.")

Fresh-load inference test passed.


## 7. HOG, GIST-style Gabor and learned VGG16 feature extraction

For each original eyebrow image:

- **HOG** captures local gradient orientation structure.
- **GIST-style descriptor** uses multi-scale, multi-orientation Gabor responses pooled over a spatial grid.
- **VGG16 feature** is the 512-dimensional global pooled representation learned from scratch.

Feature extraction is deterministic and uses no train-time augmentation.

In [14]:
def letterbox_bgr(
    image: np.ndarray,
    target_width: int,
    target_height: int,
) -> np.ndarray:
    if image is None or image.size == 0:
        raise ValueError("Cannot letterbox an empty image.")

    source_height, source_width = image.shape[:2]
    scale = min(
        target_width / source_width,
        target_height / source_height,
    )

    resized_width = max(1, int(round(source_width * scale)))
    resized_height = max(1, int(round(source_height * scale)))

    resized = cv2.resize(
        image,
        (resized_width, resized_height),
        interpolation=cv2.INTER_AREA
        if scale < 1.0
        else cv2.INTER_LINEAR,
    )

    horizontal_padding = target_width - resized_width
    vertical_padding = target_height - resized_height

    left = horizontal_padding // 2
    right = horizontal_padding - left
    top = vertical_padding // 2
    bottom = vertical_padding - top

    return cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        borderType=cv2.BORDER_REFLECT_101,
    )


def extract_hog_feature(image_bgr: np.ndarray) -> np.ndarray:
    image = letterbox_bgr(image_bgr, 128, 64)
    grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    feature = hog(
        grayscale,
        orientations=CONFIG.hog_orientations,
        pixels_per_cell=CONFIG.hog_pixels_per_cell,
        cells_per_block=CONFIG.hog_cells_per_block,
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return feature.astype(np.float32)


def extract_gist_feature(image_bgr: np.ndarray) -> np.ndarray:
    size = CONFIG.gist_image_size
    grid_size = CONFIG.gist_grid_size

    image = letterbox_bgr(image_bgr, size, size)
    grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    grayscale = grayscale.astype(np.float32) / 255.0

    grid_edges = np.linspace(0, size, grid_size + 1, dtype=int)
    orientations = np.linspace(
        0.0,
        np.pi,
        CONFIG.gist_orientations,
        endpoint=False,
    )

    features: List[float] = []

    for sigma, wavelength in CONFIG.gist_scales:
        for theta in orientations:
            kernel = cv2.getGaborKernel(
                ksize=(21, 21),
                sigma=float(sigma),
                theta=float(theta),
                lambd=float(wavelength),
                gamma=0.5,
                psi=0.0,
                ktype=cv2.CV_32F,
            )
            response = cv2.filter2D(
                grayscale,
                cv2.CV_32F,
                kernel,
            )
            response = np.abs(response)

            for row_index in range(grid_size):
                for column_index in range(grid_size):
                    y_start = grid_edges[row_index]
                    y_end = grid_edges[row_index + 1]
                    x_start = grid_edges[column_index]
                    x_end = grid_edges[column_index + 1]

                    cell = response[
                        y_start:y_end,
                        x_start:x_end,
                    ]
                    features.append(float(cell.mean()))

    return np.asarray(features, dtype=np.float32)


feature_transform = evaluation_transform


def extract_vgg_feature_from_path(image_path: Path) -> np.ndarray:
    with Image.open(image_path) as image:
        image_tensor = feature_transform(
            image.convert("RGB")
        ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feature = feature_model.extract_features(image_tensor)

    return (
        feature.squeeze(0)
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )


def l2_normalize_vector(feature: np.ndarray) -> np.ndarray:
    norm = float(np.linalg.norm(feature))
    if norm < 1e-12:
        return feature.astype(np.float32)
    return (feature / norm).astype(np.float32)


def extract_split_features(
    split_frame: pd.DataFrame,
    split_name: str,
) -> Dict[str, np.ndarray]:
    cache_path = (
        OUTPUT_DIRS["artifacts"]
        / f"features_{split_name}.npz"
    )

    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as loaded:
            cached_sample_ids = loaded["sample_ids"].astype(str)
            expected_sample_ids = (
                split_frame["sample_id"].astype(str).to_numpy()
            )

            if np.array_equal(cached_sample_ids, expected_sample_ids):
                return {
                    "hog": loaded["hog"],
                    "gist": loaded["gist"],
                    "vgg": loaded["vgg"],
                    "targets": loaded["targets"],
                    "sample_ids": cached_sample_ids,
                }

        raise RuntimeError(
            f"Existing feature cache does not match metadata: {cache_path}"
        )

    hog_features: List[np.ndarray] = []
    gist_features: List[np.ndarray] = []
    vgg_features: List[np.ndarray] = []
    targets: List[int] = []
    sample_ids: List[str] = []

    for row in tqdm(
        split_frame.itertuples(index=False),
        total=len(split_frame),
        desc=f"Extracting {split_name} features",
    ):
        image_path = Path(row.resolved_output_path)
        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )
        if image_bgr is None:
            raise RuntimeError(
                f"OpenCV could not read image: {image_path}"
            )

        hog_feature = l2_normalize_vector(
            extract_hog_feature(image_bgr)
        )
        gist_feature = l2_normalize_vector(
            extract_gist_feature(image_bgr)
        )
        vgg_feature = l2_normalize_vector(
            extract_vgg_feature_from_path(image_path)
        )

        if not np.isfinite(hog_feature).all():
            raise FloatingPointError(
                f"HOG contains NaN/Inf: {image_path}"
            )
        if not np.isfinite(gist_feature).all():
            raise FloatingPointError(
                f"GIST contains NaN/Inf: {image_path}"
            )
        if not np.isfinite(vgg_feature).all():
            raise FloatingPointError(
                f"VGG feature contains NaN/Inf: {image_path}"
            )

        hog_features.append(hog_feature)
        gist_features.append(gist_feature)
        vgg_features.append(vgg_feature)
        targets.append(int(row.target))
        sample_ids.append(str(row.sample_id))

    payload = {
        "hog": np.stack(hog_features).astype(np.float32),
        "gist": np.stack(gist_features).astype(np.float32),
        "vgg": np.stack(vgg_features).astype(np.float32),
        "targets": np.asarray(targets, dtype=np.int64),
        "sample_ids": np.asarray(sample_ids, dtype="U64"),
    }

    atomic_save_npz(cache_path, **payload)
    return payload


train_features = extract_split_features(train_frame, "train")
val_features = extract_split_features(val_frame, "val")

feature_dimensions = {
    "hog": int(train_features["hog"].shape[1]),
    "gist": int(train_features["gist"].shape[1]),
    "vgg": int(train_features["vgg"].shape[1]),
}

atomic_write_json(
    feature_dimensions,
    OUTPUT_DIRS["artifacts"] / "feature_dimensions.json",
)

print(feature_dimensions)

Extracting train features:   0%|          | 0/1562 [00:00<?, ?it/s]

Extracting val features:   0%|          | 0/204 [00:00<?, ?it/s]

{'hog': 3780, 'gist': 512, 'vgg': 512}


## 8. Train-only scaling, PCA and validation-based SVM selection

Each feature block is standardized using statistics fitted on the training split only.  
The standardized blocks are concatenated, PCA is fitted on training only, and SVM hyperparameters are selected using validation F1.

The test split is not loaded into the SVM stage until selection is complete.

In [15]:
FEATURE_BLOCK_NAMES = ("hog", "gist", "vgg")

block_scalers: Dict[str, StandardScaler] = {}
scaled_train_blocks: List[np.ndarray] = []
scaled_val_blocks: List[np.ndarray] = []

for block_name in FEATURE_BLOCK_NAMES:
    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(
        train_features[block_name]
    )
    scaled_val = scaler.transform(
        val_features[block_name]
    )

    if not np.isfinite(scaled_train).all():
        raise FloatingPointError(
            f"Scaled training block contains NaN/Inf: {block_name}"
        )
    if not np.isfinite(scaled_val).all():
        raise FloatingPointError(
            f"Scaled validation block contains NaN/Inf: {block_name}"
        )

    block_scalers[block_name] = scaler
    scaled_train_blocks.append(scaled_train.astype(np.float32))
    scaled_val_blocks.append(scaled_val.astype(np.float32))

X_train_fused = np.concatenate(
    scaled_train_blocks,
    axis=1,
)
X_val_fused = np.concatenate(
    scaled_val_blocks,
    axis=1,
)

y_train = train_features["targets"]
y_val = val_features["targets"]

maximum_pca_components = min(
    CONFIG.pca_components,
    X_train_fused.shape[0] - 1,
    X_train_fused.shape[1],
)

if maximum_pca_components < 2:
    raise ValueError(
        f"PCA component count is invalid: {maximum_pca_components}"
    )

pca = PCA(
    n_components=maximum_pca_components,
    svd_solver="randomized",
    random_state=CONFIG.seed,
)

X_train_reduced = pca.fit_transform(X_train_fused)
X_val_reduced = pca.transform(X_val_fused)

if not np.isfinite(X_train_reduced).all():
    raise FloatingPointError("PCA training output contains NaN/Inf.")
if not np.isfinite(X_val_reduced).all():
    raise FloatingPointError("PCA validation output contains NaN/Inf.")

preprocessing_artifact = {
    "feature_block_order": FEATURE_BLOCK_NAMES,
    "block_scalers": block_scalers,
    "pca": pca,
    "train_only_fit": True,
}
atomic_joblib_dump(
    preprocessing_artifact,
    OUTPUT_DIRS["artifacts"] / "feature_preprocessing.joblib",
)

atomic_write_json(
    {
        "input_dimension": int(X_train_fused.shape[1]),
        "pca_components": int(X_train_reduced.shape[1]),
        "explained_variance_ratio_sum": float(
            pca.explained_variance_ratio_.sum()
        ),
        "fitted_on_split": "train",
    },
    OUTPUT_DIRS["artifacts"] / "pca_summary.json",
)


def best_f1_threshold(
    targets: np.ndarray,
    scores: np.ndarray,
) -> Tuple[float, float]:
    precision_values, recall_values, thresholds = (
        precision_recall_curve(targets, scores)
    )

    if thresholds.size == 0:
        return 0.0, 0.0

    precision_values = precision_values[:-1]
    recall_values = recall_values[:-1]

    denominator = precision_values + recall_values
    f1_values = np.divide(
        2.0 * precision_values * recall_values,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )

    best_index = int(np.argmax(f1_values))
    return float(thresholds[best_index]), float(f1_values[best_index])


search_records: List[Dict[str, Any]] = []
candidate_models: Dict[Tuple[float, str], SVC] = {}

for c_value in CONFIG.svm_c_values:
    for gamma_value in CONFIG.svm_gamma_values:
        svm_model = SVC(
            kernel="rbf",
            C=float(c_value),
            gamma=gamma_value,
            class_weight=CONFIG.svm_class_weight,
            probability=False,
            random_state=CONFIG.seed,
        )
        svm_model.fit(X_train_reduced, y_train)

        validation_scores = svm_model.decision_function(
            X_val_reduced
        )
        threshold, validation_f1 = best_f1_threshold(
            y_val,
            validation_scores,
        )
        validation_predictions = (
            validation_scores >= threshold
        ).astype(int)
        validation_metrics = binary_metrics(
            y_val,
            validation_predictions,
            validation_scores,
        )

        gamma_key = str(gamma_value)
        record = {
            "C": float(c_value),
            "gamma": gamma_key,
            "threshold": threshold,
            **{
                f"val_{key}": value
                for key, value in validation_metrics.items()
            },
        }
        search_records.append(record)
        candidate_models[(float(c_value), gamma_key)] = svm_model

search_frame = pd.DataFrame(search_records).sort_values(
    by=["val_f1", "val_roc_auc"],
    ascending=[False, False],
).reset_index(drop=True)

best_record = search_frame.iloc[0].to_dict()
best_key = (
    float(best_record["C"]),
    str(best_record["gamma"]),
)
best_svm = candidate_models[best_key]
selected_threshold = float(best_record["threshold"])

atomic_write_csv(
    search_frame,
    OUTPUT_DIRS["metrics"] / "svm_validation_search.csv",
)
atomic_joblib_dump(
    best_svm,
    OUTPUT_DIRS["artifacts"] / "svm_model.joblib",
)
atomic_write_json(
    {
        "selected_hyperparameters": {
            "kernel": "rbf",
            "C": float(best_record["C"]),
            "gamma": str(best_record["gamma"]),
            "class_weight": CONFIG.svm_class_weight,
        },
        "selected_threshold": selected_threshold,
        "selection_split": "val",
        "selection_metric": "f1",
        "validation_metrics": {
            key.removeprefix("val_"): float(value)
            for key, value in best_record.items()
            if key.startswith("val_")
        },
    },
    OUTPUT_DIRS["metrics"] / "svm_selection.json",
)

display(search_frame.head(10))

,C,gamma,threshold,val_accuracy,val_precision,val_recall,val_f1,val_specificity,val_roc_auc,val_average_precision
0,1.0,0.0001,-0.269859,0.666667,0.629310,0.744898,0.682243,0.594340,0.666538,0.615112
1,1.0,scale,-0.793798,0.553922,0.519774,0.938776,0.669091,0.198113,0.645841,0.594431
2,0.1,0.0001,-0.636560,0.563725,0.526627,0.908163,0.666667,0.245283,0.654987,0.613000
3,0.1,scale,-0.567107,0.539216,0.511111,0.938776,0.661871,0.169811,0.638910,0.594403
4,10.0,0.0001,-1.318746,0.509804,0.494898,0.989796,0.659864,0.066038,0.623989,0.616809
5,100.0,scale,-0.686253,0.519608,0.500000,0.959184,0.657343,0.113208,0.610320,0.598644
6,10.0,scale,-0.704246,0.519608,0.500000,0.959184,0.657343,0.113208,0.610031,0.597321
7,0.1,0.001,0.770169,0.549020,0.517647,0.897959,0.656716,0.226415,0.561032,0.491684
8,1.0,0.001,-0.176116,0.549020,0.517857,0.887755,0.654135,0.235849,0.563727,0.497129
9,100.0,0.0001,-1.667681,0.490196,0.485149,1.000000,0.653333,0.018868,0.613304,0.619332


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Final test evaluation

Only after SVM selection is complete:

1. test features are extracted,
2. train-fitted scalers and PCA are applied,
3. the selected SVM and validation-selected threshold are used once,
4. frame-level and source-video-level results are saved.

In [16]:
test_features = extract_split_features(test_frame, "test")

scaled_test_blocks: List[np.ndarray] = []
for block_name in FEATURE_BLOCK_NAMES:
    transformed = block_scalers[block_name].transform(
        test_features[block_name]
    )
    if not np.isfinite(transformed).all():
        raise FloatingPointError(
            f"Scaled test block contains NaN/Inf: {block_name}"
        )
    scaled_test_blocks.append(transformed.astype(np.float32))

X_test_fused = np.concatenate(scaled_test_blocks, axis=1)
X_test_reduced = pca.transform(X_test_fused)
y_test = test_features["targets"]

test_scores = best_svm.decision_function(X_test_reduced)
test_predictions = (
    test_scores >= selected_threshold
).astype(int)

frame_metrics = binary_metrics(
    y_test,
    test_predictions,
    test_scores,
)

test_prediction_frame = test_frame[
    [
        "sample_id",
        "source_video",
        "source_video_raw",
        "source_frame_index",
        "label",
        "target",
        "resolved_output_path",
        "output_sha256",
    ]
].copy()

expected_sample_ids = test_prediction_frame[
    "sample_id"
].astype(str).to_numpy()

if not np.array_equal(
    expected_sample_ids,
    test_features["sample_ids"].astype(str),
):
    raise AssertionError(
        "Test feature order does not match test metadata order."
    )

test_prediction_frame["decision_score"] = test_scores
test_prediction_frame["decision_threshold"] = selected_threshold
test_prediction_frame["prediction"] = test_predictions
test_prediction_frame["predicted_label"] = test_prediction_frame[
    "prediction"
].map({0: "real", 1: "fake"})
test_prediction_frame["correct"] = (
    test_prediction_frame["target"]
    == test_prediction_frame["prediction"]
)

label_consistency = (
    test_prediction_frame.groupby("source_video")["target"].nunique()
)
if (label_consistency > 1).any():
    raise AssertionError(
        "At least one source video contains conflicting labels."
    )

video_prediction_frame = (
    test_prediction_frame.groupby(
        ["source_video", "source_video_raw"],
        as_index=False,
    )
    .agg(
        target=("target", "first"),
        label=("label", "first"),
        mean_decision_score=("decision_score", "mean"),
        median_decision_score=("decision_score", "median"),
        frame_count=("sample_id", "count"),
    )
)

video_prediction_frame["decision_threshold"] = selected_threshold
video_prediction_frame["prediction"] = (
    video_prediction_frame["mean_decision_score"]
    >= selected_threshold
).astype(int)
video_prediction_frame["predicted_label"] = (
    video_prediction_frame["prediction"]
    .map({0: "real", 1: "fake"})
)
video_prediction_frame["correct"] = (
    video_prediction_frame["target"]
    == video_prediction_frame["prediction"]
)

video_metrics = binary_metrics(
    video_prediction_frame["target"].to_numpy(),
    video_prediction_frame["prediction"].to_numpy(),
    video_prediction_frame["mean_decision_score"].to_numpy(),
)

atomic_write_csv(
    test_prediction_frame,
    OUTPUT_DIRS["predictions"]
    / "test_predictions_frame_level.csv",
)
atomic_write_csv(
    video_prediction_frame,
    OUTPUT_DIRS["predictions"]
    / "test_predictions_video_level.csv",
)
atomic_write_json(
    frame_metrics,
    OUTPUT_DIRS["metrics"] / "test_metrics_frame_level.json",
)
atomic_write_json(
    video_metrics,
    OUTPUT_DIRS["metrics"] / "test_metrics_video_level.json",
)

metrics_table = pd.DataFrame(
    [
        {"evaluation_level": "frame", **frame_metrics},
        {"evaluation_level": "video", **video_metrics},
    ]
)
atomic_write_csv(
    metrics_table,
    OUTPUT_DIRS["metrics"] / "test_metrics_summary.csv",
)

display(metrics_table)

Extracting test features:   0%|          | 0/196 [00:00<?, ?it/s]

,evaluation_level,accuracy,precision,recall,f1,specificity,roc_auc,average_precision
0,frame,0.530612,0.513761,0.589474,0.549020,0.475248,0.544971,0.553526
1,video,0.521739,0.597403,0.567901,0.582278,0.456140,0.545809,0.655939


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
def plot_confusion_matrix(
    targets: np.ndarray,
    predictions: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    matrix = confusion_matrix(
        targets,
        predictions,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(8, 8), dpi=CONFIG.figure_dpi)
    image = ax.imshow(matrix)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            ax.text(
                column_index,
                row_index,
                str(matrix[row_index, column_index]),
                ha="center",
                va="center",
                fontsize=13,
            )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("True Class", fontsize=11)
    ax.set_xticks([0, 1], labels=["Real", "Fake"])
    ax.set_yticks([0, 1], labels=["Real", "Fake"])
    fig.colorbar(image, ax=ax)
    save_figure(fig, output_path)


plot_confusion_matrix(
    test_prediction_frame["target"].to_numpy(),
    test_prediction_frame["prediction"].to_numpy(),
    "Frame-Level Test Confusion Matrix",
    OUTPUT_DIRS["figures"]
    / "test_confusion_matrix_frame_level.png",
)

plot_confusion_matrix(
    video_prediction_frame["target"].to_numpy(),
    video_prediction_frame["prediction"].to_numpy(),
    "Video-Level Test Confusion Matrix",
    OUTPUT_DIRS["figures"]
    / "test_confusion_matrix_video_level.png",
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
def plot_roc(
    targets: np.ndarray,
    scores: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    false_positive_rate, true_positive_rate, _ = roc_curve(
        targets,
        scores,
    )
    area = roc_auc_score(targets, scores)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"ROC AUC = {area:.4f}",
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate", fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.25)
    save_figure(fig, output_path)


def plot_precision_recall(
    targets: np.ndarray,
    scores: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    precision_values, recall_values, _ = precision_recall_curve(
        targets,
        scores,
    )
    area = average_precision_score(targets, scores)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)
    ax.plot(
        recall_values,
        precision_values,
        linewidth=2,
        label=f"Average Precision = {area:.4f}",
    )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.25)
    save_figure(fig, output_path)


plot_roc(
    test_prediction_frame["target"].to_numpy(),
    test_prediction_frame["decision_score"].to_numpy(),
    "Frame-Level Test ROC Curve",
    OUTPUT_DIRS["figures"] / "test_roc_curve_frame_level.png",
)

plot_roc(
    video_prediction_frame["target"].to_numpy(),
    video_prediction_frame["mean_decision_score"].to_numpy(),
    "Video-Level Test ROC Curve",
    OUTPUT_DIRS["figures"] / "test_roc_curve_video_level.png",
)

plot_precision_recall(
    test_prediction_frame["target"].to_numpy(),
    test_prediction_frame["decision_score"].to_numpy(),
    "Frame-Level Test Precision-Recall Curve",
    OUTPUT_DIRS["figures"]
    / "test_precision_recall_curve_frame_level.png",
)

plot_precision_recall(
    video_prediction_frame["target"].to_numpy(),
    video_prediction_frame["mean_decision_score"].to_numpy(),
    "Video-Level Test Precision-Recall Curve",
    OUTPUT_DIRS["figures"]
    / "test_precision_recall_curve_video_level.png",
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
# Decision score distribution, saved as a separate high-resolution figure.
fig, ax = plt.subplots(figsize=(10, 6), dpi=CONFIG.figure_dpi)

real_scores = test_prediction_frame.loc[
    test_prediction_frame["target"] == 0,
    "decision_score",
]
fake_scores = test_prediction_frame.loc[
    test_prediction_frame["target"] == 1,
    "decision_score",
]

ax.hist(real_scores, bins=20, alpha=0.65, label="Real")
ax.hist(fake_scores, bins=20, alpha=0.65, label="Fake")
ax.axvline(
    selected_threshold,
    linestyle="--",
    linewidth=2,
    label=f"Decision Threshold = {selected_threshold:.4f}",
)
ax.set_title("Frame-Level Test Decision Score Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("SVM Decision Score", fontsize=11)
ax.set_ylabel("Number of Samples", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(
    fig,
    OUTPUT_DIRS["figures"]
    / "test_decision_score_distribution.png",
)

## 10. Final output audit and reproducibility package

This cell:

- verifies every generated figure,
- records file hashes,
- captures the exact Python environment,
- writes a final run summary,
- and creates a complete output manifest.

In [20]:
# Capture the exact runtime versions used for this experiment.
requirements_text = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
atomic_write_text(
    requirements_text,
    RUN_DIR / "requirements_lock.txt",
)

environment_payload = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}
atomic_write_json(
    environment_payload,
    RUN_DIR / "environment.json",
)

figure_audit_records: List[Dict[str, Any]] = []
for figure_path in sorted(OUTPUT_DIRS["figures"].glob("*")):
    if figure_path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
        continue

    with Image.open(figure_path) as image:
        width, height = image.size

    if min(width, height) < CONFIG.minimum_figure_short_edge_px:
        raise AssertionError(
            f"Figure failed final resolution audit: "
            f"{figure_path} -> {(width, height)}"
        )

    figure_audit_records.append(
        {
            "file_name": figure_path.name,
            "width_px": width,
            "height_px": height,
            "short_edge_px": min(width, height),
            "sha256": sha256_file(figure_path),
            "language": "English",
        }
    )

atomic_write_csv(
    pd.DataFrame(figure_audit_records),
    OUTPUT_DIRS["metrics"] / "figure_quality_audit.csv",
)

manifest_records: List[Dict[str, Any]] = []
for file_path in sorted(
    path
    for path in RUN_DIR.rglob("*")
    if path.is_file()
):
    manifest_records.append(
        {
            "relative_path": str(file_path.relative_to(RUN_DIR)),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256_file(file_path),
        }
    )

atomic_write_csv(
    pd.DataFrame(manifest_records),
    RUN_DIR / "output_manifest.csv",
)

run_summary = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "model": {
        "cnn": "VGG16 trained from scratch",
        "pretrained_weights": False,
        "feature_fusion": ["HOG", "GIST-style Gabor", "VGG16 learned features"],
        "classifier": "RBF SVM",
    },
    "paths": {
        "input_data": str(DATA_ROOT),
        "output_run": str(RUN_DIR),
        "best_checkpoint": str(best_checkpoint_path),
        "svm_model": str(
            OUTPUT_DIRS["artifacts"] / "svm_model.joblib"
        ),
    },
    "data": accounting_payload,
    "selected_svm": {
        "C": float(best_record["C"]),
        "gamma": str(best_record["gamma"]),
        "threshold": selected_threshold,
    },
    "test_metrics": {
        "frame_level": frame_metrics,
        "video_level": video_metrics,
    },
    "training_duration_seconds": training_duration_seconds,
    "figure_count": len(figure_audit_records),
    "completed_at": datetime.now().isoformat(),
}

atomic_write_json(
    run_summary,
    RUN_DIR / "run_summary.json",
)

print("=" * 80)
print("EXPERIMENT COMPLETED")
print(f"Run ID: {RUN_ID}")
print(f"All outputs were saved to: {RUN_DIR}")
print("=" * 80)
print(json.dumps(run_summary["test_metrics"], indent=2))

EXPERIMENT COMPLETED
Run ID: 20260806_1540_eyebrow_vgg16scratch_hog_gist_svm_seed42
All outputs were saved to: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_1540_eyebrow_vgg16scratch_hog_gist_svm_seed42
{
  "frame_level": {
    "accuracy": 0.5306122448979592,
    "precision": 0.5137614678899083,
    "recall": 0.5894736842105263,
    "f1": 0.5490196078431373,
    "specificity": 0.4752475247524752,
    "roc_auc": 0.5449713392391871,
    "average_precision": 0.5535261514531744
  },
  "video_level": {
    "accuracy": 0.5217391304347826,
    "precision": 0.5974025974025974,
    "recall": 0.5679012345679012,
    "f1": 0.5822784810126582,
    "specificity": 0.45614035087719296,
    "roc_auc": 0.5458089668615984,
    "average_precision": 0.6559391938208657
  }
}


## Expected output structure

```text
Sonuçlar/
└── <run_id>/
    ├── config_resolved.yaml
    ├── requirements_lock.txt
    ├── environment.json
    ├── output_manifest.csv
    ├── run_summary.json
    ├── checkpoints/
    │   ├── best.ckpt
    │   ├── last.ckpt
    │   └── epoch_N.ckpt
    ├── logs/
    │   └── training_history.csv
    ├── metrics/
    │   ├── data_accounting.json
    │   ├── quality_gates.json
    │   ├── svm_validation_search.csv
    │   ├── svm_selection.json
    │   ├── test_metrics_frame_level.json
    │   ├── test_metrics_video_level.json
    │   ├── test_metrics_summary.csv
    │   └── figure_quality_audit.csv
    ├── predictions/
    │   ├── test_predictions_frame_level.csv
    │   └── test_predictions_video_level.csv
    ├── figures/
    │   ├── dataset_distribution.png
    │   ├── cnn_loss_curve.png
    │   ├── cnn_accuracy_curve.png
    │   ├── cnn_f1_curve.png
    │   ├── test_confusion_matrix_*.png
    │   ├── test_roc_curve_*.png
    │   ├── test_precision_recall_curve_*.png
    │   └── test_decision_score_distribution.png
    └── artifacts/
        ├── metadata_used.csv
        ├── train_normalization.json
        ├── model_architecture.json
        ├── features_train.npz
        ├── features_val.npz
        ├── features_test.npz
        ├── feature_dimensions.json
        ├── feature_preprocessing.joblib
        ├── pca_summary.json
        └── svm_model.joblib
```